# fig1_new_region - Part 1/8\n\n自动拆分版本（按步骤执行）。\n包含统一 bootstrap 和共享模块导入。\n

In [ ]:
# AUTO_BOOTSTRAP_V2
from pathlib import Path
import sys
import os
import builtins
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "projects").exists():
    cur = Path.cwd().resolve()
    for p in [cur] + list(cur.parents):
        if (p / "projects").exists():
            ROOT = p
            break

NB_PATH = Path.cwd()
if "clone_motif" in str(NB_PATH):
    PROJECT_DIR = ROOT / "projects" / "clone_motif"
else:
    PROJECT_DIR = ROOT / "projects" / "our_multiregion_motif"

DATA_RAW = PROJECT_DIR / "data" / "raw"
DATA_PROCESSED = PROJECT_DIR / "data" / "processed"
OUT_FIG = PROJECT_DIR / "outputs" / "figures"
OUT_TABLE = PROJECT_DIR / "outputs" / "tables"
OUT_ARCH = PROJECT_DIR / "outputs" / "archives"

for d in [DATA_PROCESSED, OUT_FIG, OUT_TABLE, OUT_ARCH]:
    d.mkdir(parents=True, exist_ok=True)

SHARED_SRC = ROOT / "projects" / "shared" / "src"
if str(SHARED_SRC) not in sys.path:
    sys.path.append(str(SHARED_SRC))

from motif_common import combination, indices_for_region, union_indices_for_regions, p_to_star, format_p_decimal_3sig, sort_by_order, truncate_colormap

READ_EXT = {".csv", ".json", ".npy", ".pkl", ".xlsx"}
FIG_EXT = {".svg", ".png", ".pdf"}
TABLE_EXT = {".csv", ".xlsx"}


def _as_path(x):
    return Path(x) if isinstance(x, (str, os.PathLike)) else x


def resolve_read_path(path):
    p = _as_path(path)
    if not isinstance(p, Path):
        return path
    if p.is_absolute() or p.exists():
        return str(p)
    if p.suffix.lower() in READ_EXT:
        for c in [DATA_RAW / p.name, ROOT / p.name]:
            if c.exists():
                return str(c)
    return str(p)


def resolve_write_path(path):
    p = _as_path(path)
    if not isinstance(p, Path):
        return path
    if p.is_absolute():
        p.parent.mkdir(parents=True, exist_ok=True)
        return str(p)
    ext = p.suffix.lower()
    if ext in FIG_EXT:
        out = OUT_FIG / p.name
    elif ext in TABLE_EXT:
        out = OUT_TABLE / p.name
    elif ext == ".zip":
        out = OUT_ARCH / p.name
    elif ext == ".npy":
        out = DATA_PROCESSED / p.name
    else:
        out = PROJECT_DIR / p
    out.parent.mkdir(parents=True, exist_ok=True)
    return str(out)

if not hasattr(builtins, "_orig_open_codex"):
    builtins._orig_open_codex = builtins.open


def _open_patch(file, mode="r", *args, **kwargs):
    if isinstance(file, (str, os.PathLike)):
        if any(m in mode for m in ["r", "a"]):
            file = resolve_read_path(file)
        if any(m in mode for m in ["w", "a", "x"]):
            file = resolve_write_path(file)
    return builtins._orig_open_codex(file, mode, *args, **kwargs)


builtins.open = _open_patch

if not hasattr(np, "_orig_load_codex"):
    np._orig_load_codex = np.load
np.load = lambda file, *a, **k: np._orig_load_codex(resolve_read_path(file), *a, **k)

if not hasattr(np, "_orig_save_codex"):
    np._orig_save_codex = np.save
np.save = lambda file, arr, *a, **k: np._orig_save_codex(resolve_write_path(file), arr, *a, **k)

if not hasattr(pd, "_orig_read_csv_codex"):
    pd._orig_read_csv_codex = pd.read_csv
pd.read_csv = lambda f, *a, **k: pd._orig_read_csv_codex(resolve_read_path(f) if isinstance(f, (str, os.PathLike)) else f, *a, **k)

if not hasattr(pd, "_orig_read_excel_codex"):
    pd._orig_read_excel_codex = pd.read_excel
pd.read_excel = lambda f, *a, **k: pd._orig_read_excel_codex(resolve_read_path(f) if isinstance(f, (str, os.PathLike)) else f, *a, **k)

if not hasattr(pd.DataFrame, "_orig_to_csv_codex"):
    pd.DataFrame._orig_to_csv_codex = pd.DataFrame.to_csv


def _to_csv_patch(self, path_or_buf=None, *args, **kwargs):
    if isinstance(path_or_buf, (str, os.PathLike)):
        path_or_buf = resolve_write_path(path_or_buf)
    return pd.DataFrame._orig_to_csv_codex(self, path_or_buf, *args, **kwargs)


pd.DataFrame.to_csv = _to_csv_patch

if not hasattr(pd.DataFrame, "_orig_to_excel_codex"):
    pd.DataFrame._orig_to_excel_codex = pd.DataFrame.to_excel


def _to_excel_patch(self, excel_writer, *args, **kwargs):
    if isinstance(excel_writer, (str, os.PathLike)):
        excel_writer = resolve_write_path(excel_writer)
    return pd.DataFrame._orig_to_excel_codex(self, excel_writer, *args, **kwargs)


pd.DataFrame.to_excel = _to_excel_patch

try:
    import matplotlib.pyplot as plt
    if not hasattr(plt, "_orig_savefig_codex"):
        plt._orig_savefig_codex = plt.savefig
    plt.savefig = lambda f, *a, **k: plt._orig_savefig_codex(resolve_write_path(f) if isinstance(f, (str, os.PathLike)) else f, *a, **k)
except Exception:
    pass

print(f"[bootstrap] project={PROJECT_DIR.name} data={DATA_RAW}")


In [ ]:
import json, re
import numpy as np




In [ ]:
##motif ER 期望和标准差计算示例
import math
import itertools
import numpy as np

def motif_params():
    """
    返回有向 3‑节点 motif 的 (a_j, g_j) 列表，顺序对应常见的 13 种 motif。
    这里给出常用的 13 种（参考文献中的自同构数）。
    """
    # (a_j, g_j) 参考自文献（Milo et al., Science 2002）和 Picard et al. 2006
    return [
        (3, 2),  # 2‑edge chain (type 1)
        (3, 2),  # 2‑edge chain (type 2)
        (6, 2),  # 2‑edge chain (type 3)
        (6, 3),  # 3‑edge fan (type 4)
        (6, 3),  # 3‑edge fan (type 5)
        (6, 3),  # 3‑edge fan (type 6)
        (3, 4),  # 4‑edge motif (type 7)
        (3, 4),  # 4‑edge motif (type 8)
        (3, 4),  # 4‑edge motif (type 9)
        (2, 3),  # 3‑edge directed triangle (type 10)
        (6, 4),  # 4‑edge motif (type 11)
        (6, 5),  # 5‑edge motif (type 12)
        (1, 6),  # 6‑edge (complete digraph) (type 13)
    ]


def expected_and_std(N, p, use_overlap=False):
    n = math.comb(N, 3)              
    results = []
    for a, g in motif_params():
        pi = a * (p ** g) * ((1 - p) ** (6 - g))
        mu = n * pi
        var = n * pi * (1 - pi)
        std = math.sqrt(var) if var > 0 else 0.0
        results.append((mu, std))

    return results


def expected_motif_mu_sigma_from_edges(N, edge, use_overlap=False):
    total_possible = N * (N - 1)          # 有向边最大可能数
    p = edge / total_possible if total_possible > 0 else 0.0

    mu_std = expected_and_std(N, p, use_overlap=use_overlap)
    mu  = np.array([m for (m, s) in mu_std], dtype=float)
    std = np.array([s for (m, s) in mu_std], dtype=float)
    return mu, std

# ------------------- 示例调用 -------------------
if __name__ == "__main__":
    N = 579          # 节点数
    total_possible = N * (N - 1)
    edge = 22908  # 有向边数
    
    # print(p)
    mu_std = expected_motif_mu_sigma_from_edges(N, edge, use_overlap=False)

    print(mu_std)


In [ ]:
import pickle
import numpy as np
import torch

# 你自己的类 / 函数
# from your_module import motifRegular, expected_motif_mu_sigma_from_edges

# 1. 读取 SC 结果
with open("wb_alltype_sc_results_dict_1110.pkl", "rb") as f:
    sc_pack = pickle.load(f)

results_sc   = sc_pack["results"][5]

In [ ]:
import json, time
import numpy as np
import networkx as nx
import pandas as pd
import math
import torch
from tqdm import tqdm
import pickle



class motifRegular:
    def __init__(self, device="cpu", numOfNeuron=512, amplitude=1000, bias=0.05):
        if isinstance(device, str):
            device = torch.device(device)
        
        self.L = torch.ones([1, numOfNeuron]).to(device)
        self.I = torch.zeros([numOfNeuron, numOfNeuron]).to(device)
        self.P = torch.zeros([numOfNeuron, numOfNeuron]).to(device)
        self.obs = torch.zeros([14]).to(device)
        self.sum = combination(numOfNeuron, 3)
        self.recordSum = 0
        self.amplitude = amplitude
        self.bias = bias
        self.device = device

        for i in range(numOfNeuron):
            self.I[i][i] = 1
        for i in range(numOfNeuron):
            for j in range(numOfNeuron):
                if i == j:
                    continue
                self.P[i][j] = 1

    def cal(self, a):
        w = torch.where(a > 0.0, torch.ones_like(a), torch.zeros_like(a))
        w = w * self.P
        pmw = self.P - w

        w0 = pmw * pmw.T
        w1 = w   * pmw.T
        w2 = pmw * w.T
        w3 = w   * w.T

        q = torch.zeros([14], dtype=torch.float32, device=self.device)

        q[1]  = 0.5 * self.L @ (w1 * (w1 @ w0)) @ self.L.T
        q[2]  = 0.5 * self.L @ (w0 * (w1 @ w2)) @ self.L.T
        q[3]  =       self.L @ (w1 * (w0 @ w2)) @ self.L.T
        q[4]  =       self.L @ (w1 * (w1 @ w2)) @ self.L.T

        q[5]  =       self.L @ (w3 * (w1 @ w0)) @ self.L.T
        q[6]  =       self.L @ (w3 * (w2 @ w0)) @ self.L.T
        q[7]  = 0.5 * self.L @ (w3 * (w1 @ w2)) @ self.L.T
        q[8]  = 0.5 * self.L @ (w3 * (w2 @ w1)) @ self.L.T

        q[9]  = 0.5 * self.L @ (w3 * (w3 @ w0)) @ self.L.T
        q[10] = (1.0/3.0) * self.L @ (w1 * (w2 @ w2)) @ self.L.T
        q[11] =       self.L @ (w3 * (w2 @ w2)) @ self.L.T
        q[12] =       self.L @ (w3 * (w3 @ w2)) @ self.L.T
        q[13] = (1.0/6.0) * self.L @ (w3 * (w3 @ w3)) @ self.L.T

        return q[1:14]


def random_er_adj_np(n, e, rng):
    """
    返回一个 n×n 的 numpy 矩阵 A，A[i,j]∈{0,1}，
    无自环，恰好 e 条有向边。
    """
    max_edges = n * (n - 1)
    if e > max_edges:
        raise ValueError(f"要求边数 e={e} > 最大可能边数 {max_edges}")

    # 所有非对角线位置的布尔 mask
    mask = np.ones((n, n), dtype=bool)
    np.fill_diagonal(mask, False)

    # 所有 off-diagonal 的一维索引
    all_idx = np.flatnonzero(mask)   # 长度 n*(n-1)

    # 随机选 e 个位置
    chosen = rng.choice(all_idx, size=e, replace=False)

    # 构建邻接矩阵
    A = np.zeros((n, n), dtype=np.float32)
    A.flat[chosen] = 1.0
    return A

def sample_random_counts(n, e, n_rand, seed=None, device="cpu"):
    rng = np.random.default_rng(seed)
    samples = np.zeros((n_rand, 13), dtype=float)
    t_gen_total   = 0.0   # 生成随机图的总时间
    t_motif_total = 0.0   # triad_counts_connected 的总时间
    mr = motifRegular(device=device, numOfNeuron=n)

    mask = np.ones((n, n), dtype=bool)
    np.fill_diagonal(mask, False)
    offdiag_idx = np.flatnonzero(mask)

    A = np.zeros((n, n), dtype=np.float32)
    for r in tqdm(range(n_rand), desc="Sampling ER", leave=False):

        # 复用 A，只重置
        A.fill(0.0)
        chosen = rng.choice(offdiag_idx, size=e, replace=False)
        A.flat[chosen] = 1.0

        # 直接用 mr.cal
        q = mr.cal(torch.from_numpy(A).to(device=device))
        samples[r] = q.cpu().numpy()

    return samples

def analyze_and_plot(A, p=1.0, n_rand=100, threshold=0.0, seed=0, device="cpu"):
    A = np.array(A, dtype=float)
    np.fill_diagonal(A, 0.0)

    # 真实网络
    A = np.array(A, dtype=float)
    n_real = A.shape[0]

    # 边的布尔矩阵：大于 threshold 的都算边
    edge_mask = A > threshold

    # 去掉自环
    np.fill_diagonal(edge_mask, False)

    e_real = int(edge_mask.sum())
    p_real = e_real / (n_real * (n_real - 1))

    # 按 p 缩小网络规模，只在 ER 采样时用
    if p < 1.0:
        n_sub = int(round(n_real * p))
        if n_sub < 2:
            raise ValueError("p 太小导致 n_sub < 2")
        e_sub = int(round(p_real * (n_sub * (n_sub - 1))))
    else:
        n_sub = n_real
        e_sub = e_real

    # 随机网络采样
    samples = sample_random_counts(n_sub, e_sub, n_rand=n_rand, seed=seed, device=device)
    mu = samples.mean(axis=0)
    sd = samples.std(axis=0, ddof=1)

    # 结果表格
    idx = [f"Motif {i+1}" for i in range(13)]
    df = pd.DataFrame({
        "ER mean": mu,
        "ER sd":   sd,
    }, index=idx)

    return {
        "table":  df,
        "er_mu":  mu,
        "er_sd":  sd,
        "n":      n_sub,
        "e":      e_sub,
        "samples": samples,   # shape: (n_rand, 13)
    }



In [ ]:
import numpy as np
import pandas as pd
import torch
import pickle
from tqdm import tqdm

# ===== 你已有：motifRegular / sample_random_counts（原样保留）=====
# （这里假设你已经把 motifRegular / sample_random_counts 粘贴在上面了）

# ================== 配置 ==================
pkl_path = "wb_alltype_sc_results_dict_1209_3groups.pkl"  # 你的 3组结果
groups = ["MOp", "MOs", "PFC"]

edge_thr = 0.0      # sc > edge_thr 认为有边（一般用 0）
n_rand = 2000
seed = 42
device = "cpu"

# ================== 读 pkl ==================
d = pickle.load(open(pkl_path, "rb"))
results = d["results"]
threshold_set = d.get("threshold_set", sorted(results.keys()))

# ================== 主循环：每个阈值、每个脑区算 Z / NZ ==================
out_rows = []
nz_dict = {}  # nz_dict[thr][group] = (13,) 方便后面画图/对比

for thr in threshold_set:
    nz_dict[thr] = {}
    for g in groups:
        if g not in results[thr]:
            print(f"[SKIP] thr={thr} group={g} not found in pkl")
            continue

        A = np.asarray(results[thr][g]["sc"], float)
        np.fill_diagonal(A, 0.0)

        edge = (A > edge_thr)
        np.fill_diagonal(edge, False)

        n = A.shape[0]
        e = int(edge.sum())
        if n < 3 or e == 0:
            print(f"[SKIP] thr={thr} group={g}: n={n}, e={e}")
            continue

        # ---- real motif（13维）----
        mr = motifRegular(device=device, numOfNeuron=n)
        real = mr.cal(torch.from_numpy(edge.astype(np.float32)).to(device)).cpu().numpy()

        # ---- ER 采样（固定 n,e）----
        samples = sample_random_counts(n, e, n_rand=n_rand, seed=seed, device=device)
        mu = samples.mean(axis=0)
        sd = samples.std(axis=0, ddof=1)

        # ---- Z / NZ ----
        z = np.zeros_like(real)
        ok = sd > 1e-8
        z[ok] = (real[ok] - mu[ok]) / sd[ok]

        znorm = np.linalg.norm(z)
        nz = z / znorm if znorm > 1e-12 else z * 0.0  # 全0时避免 NaN

        nz_dict[thr][g] = nz

        # ---- 记录成表（长表：每行=thr+group+motif）----
        for k in range(13):
            out_rows.append({
                "distance_thr": thr,      # 你 KDTree 半径阈值
                "group": g,               # MOp/MOs/PFC
                "motif": f"Motif {k+1}",
                "n": n,
                "e": e,
                "sparsity": e / (n * (n - 1)),
                "real": float(real[k]),
                "ER_mean": float(mu[k]),
                "ER_sd": float(sd[k]),
                "Z_ER": float(z[k]),
                "NZ_ER": float(nz[k]),
            })

        print(f"[OK] thr={thr} group={g} n={n} e={e}")

# ================== 保存结果 ==================
df_out = pd.DataFrame(out_rows)
df_out.to_csv("NZ_3groups_by_distanceThr.csv", index=False)
print("\n✅ Saved: NZ_3groups_by_distanceThr.csv")

# ================== 可选：每个 thr 输出一个 3×13 的 NZ 表（便于看/画） ==================
for thr in threshold_set:
    if thr not in nz_dict:
        continue
    rows = []
    for g in groups:
        if g in nz_dict[thr]:
            rows.append(pd.Series(nz_dict[thr][g], index=[f"M{i}" for i in range(1,14)], name=g))
    if rows:
        wide = pd.DataFrame(rows)
        wide.to_csv(f"NZ_{thr}_3groups_wide.csv")
        print(f"✅ Saved: NZ_{thr}_3groups_wide.csv")

In [ ]:
if __name__ == "__main__":
    # ===== 1. 路径自己改一下 =====
    sc_path    = "wb_alltype_sc_subset_zj_1_3_42_1209.npy"          # 你的 sc 矩阵
    names_path = "wb_alltype_sc_subset_names_zj_1_3_42_1209.npy"    # 对应的 neuron 名字
    csv_path   = "Metadata_PFC_MOp.csv"                             # meta 信息

    # ===== 2. 载入数据 =====
    sc = np.load(sc_path)                            # N x N
    names = np.load(names_path, allow_pickle=True)   # 长度 N
    if isinstance(names, np.ndarray):
        names = names.tolist()

    df = pd.read_csv(csv_path)

    # 猜一下哪一列对应 names
    if "name" in df.columns:
        neuron_id_col = "name"
    elif "Unnamed: 0" in df.columns:
        neuron_id_col = "Unnamed: 0"
    else:
        raise ValueError("CSV 里找不到和 names 对应的列（尝试了 'name' / 'Unnamed: 0'），需要手动指定 neuron_id_col。")

    if "clone" not in df.columns:
        raise ValueError("CSV 里没有 'clone' 列，无法按 clone 分组。")

    # ===== 3. 确定哪个列代表 MOp（优先用 SomaRegion）=====
    if "SomaRegion" in df.columns:
        region_col = "SomaRegion"
    elif "Region" in df.columns:
        region_col = "Region"
    else:
        raise ValueError("CSV 里既没有 'SomaRegion' 也没有 'Region'，不知道从哪一列判断 MOp。")

    # 这些 neuron 是在 MOp 的（比如 SomaRegion == 'MOp'）
    mask_mop = df[region_col].astype(str) == "MOp"
    df_mop   = df[mask_mop].copy()

    if df_mop.empty:
        raise ValueError("在 CSV 里没有找到任何 Region/SomaRegion 为 MOp 的神经元。")

    # 含有 MOp 神经元的 clone 集合
    mop_clones = set(df_mop["clone"].astype(str).unique())
    print(f"含 MOp 神经元的 clone 个数: {len(mop_clones)}")
    print("示例几个 clone:", list(mop_clones)[:10])

    # ===== 4. 建立 name -> clone 映射 =====
    df["__name_str__"] = df[neuron_id_col].astype(str)
    name_to_clone = dict(zip(df["__name_str__"], df["clone"].astype(str)))

    # ===== 5. 在 names 里找到：属于这些 clone 的所有节点索引 =====
    idxs = []
    for i, nid in enumerate(names):
        nid_str = str(nid)
        cl = name_to_clone.get(nid_str, None)
        if cl in mop_clones:
            idxs.append(i)

    print(f"这些含 MOp 的 clone 一共包含神经元数: {len(idxs)}")

    if len(idxs) < 3:
        raise ValueError("整体节点数 < 3，没法算 triad motif。")

    # ===== 6. 取子矩阵，算真实 motif + ER baseline =====
    A_sub = sc[np.ix_(idxs, idxs)]
    threshold = 0.0   # >0 看作一条边，你如果之前有别的阈值就同步改

    # 真实网络：二值邻接矩阵
    W_bin = (A_sub > threshold).astype(np.float32)
    np.fill_diagonal(W_bin, 0.0)
    n = W_bin.shape[0]
    e = int((W_bin > 0).sum())
    max_edges = n * (n - 1)
    sparsity = e / max_edges if max_edges > 0 else np.nan

    print("\n=== 含 MOp 的 clone 合并子图：基本信息 ===")
    print("节点数 n =", n)
    print("边数 e  =", e)
    print("稀疏度 sparsity =", sparsity)

    # ---- 真实 motif 计数 ----
    mr_real = motifRegular(device="cpu", numOfNeuron=n)
    real_counts = mr_real.cal(torch.from_numpy(W_bin)).cpu().numpy()  # shape (13,)

    # ---- ER baseline：使用你原来的 analyze_and_plot ----
    er_res = analyze_and_plot(
        A_sub,
        p=1.0,
        n_rand=200,
        threshold=threshold,
        seed=42,
        device="cpu",
    )
    mu = er_res["er_mu"]
    sd = er_res["er_sd"]

    # ---- 计算 NZ-score ----
    z = np.zeros_like(real_counts)
    valid = sd > 1e-8
    z[valid] = (real_counts[valid] - mu[valid]) / sd[valid]

    motif_labels = [f"Motif {i+1}" for i in range(13)]
    df_motif = pd.DataFrame({
        "real": real_counts,
        "ER_mean": mu,
        "ER_sd": sd,
        "NZ_ER": z,
    }, index=motif_labels)

    print("\n=== 含 MOp 的 clone（所有相关神经元合在一起）的 motif 分布 ===")
    print(df_motif.round(2))

    # 如果你想存一下：
    df_motif.to_csv("motif_MOp_related_clones.csv")

In [ ]:
import numpy as np
import pandas as pd
import math
import torch
from tqdm import tqdm

# ================= motifRegular 和 ER 随机采样 =================


class motifRegular:
    def __init__(self, device="cpu", numOfNeuron=512, amplitude=1000, bias=0.05):
        if isinstance(device, str):
            device = torch.device(device)
        
        self.L = torch.ones([1, numOfNeuron]).to(device)
        self.I = torch.zeros([numOfNeuron, numOfNeuron]).to(device)
        self.P = torch.zeros([numOfNeuron, numOfNeuron]).to(device)
        self.obs = torch.zeros([14]).to(device)
        self.sum = combination(numOfNeuron, 3)
        self.recordSum = 0
        self.amplitude = amplitude
        self.bias = bias
        self.device = device

        for i in range(numOfNeuron):
            self.I[i][i] = 1
        for i in range(numOfNeuron):
            for j in range(numOfNeuron):
                if i == j:
                    continue
                self.P[i][j] = 1

    def cal(self, a):
        # a: n x n torch tensor
        w = torch.where(a > 0.0, torch.ones_like(a), torch.zeros_like(a))
        w = w * self.P
        pmw = self.P - w

        w0 = pmw * pmw.T
        w1 = w   * pmw.T
        w2 = pmw * w.T
        w3 = w   * w.T

        q = torch.zeros([14], dtype=torch.float32, device=self.device)

        q[1]  = 0.5 * self.L @ (w1 * (w1 @ w0)) @ self.L.T
        q[2]  = 0.5 * self.L @ (w0 * (w1 @ w2)) @ self.L.T
        q[3]  =       self.L @ (w1 * (w0 @ w2)) @ self.L.T
        q[4]  =       self.L @ (w1 * (w1 @ w2)) @ self.L.T

        q[5]  =       self.L @ (w3 * (w1 @ w0)) @ self.L.T
        q[6]  =       self.L @ (w3 * (w2 @ w0)) @ self.L.T
        q[7]  = 0.5 * self.L @ (w3 * (w1 @ w2)) @ self.L.T
        q[8]  = 0.5 * self.L @ (w3 * (w2 @ w1)) @ self.L.T

        q[9]  = 0.5 * self.L @ (w3 * (w3 @ w0)) @ self.L.T
        q[10] = (1.0/3.0) * self.L @ (w1 * (w2 @ w2)) @ self.L.T
        q[11] =       self.L @ (w3 * (w2 @ w2)) @ self.L.T
        q[12] =       self.L @ (w3 * (w3 @ w2)) @ self.L.T
        q[13] = (1.0/6.0) * self.L @ (w3 * (w3 @ w3)) @ self.L.T

        return q[1:14]  # 13 motifs


def sample_random_counts(n, e, n_rand, seed=None, device="cpu"):
    """
    对给定节点数 n、边数 e 的 ER 图，采样 n_rand 个随机图，
    用 motifRegular 计算每个图上的 triad motif 计数。
    返回 shape = (n_rand, 13)
    """
    rng = np.random.default_rng(seed)
    samples = np.zeros((n_rand, 13), dtype=float)

    mask = np.ones((n, n), dtype=bool)
    np.fill_diagonal(mask, False)
    offdiag_idx = np.flatnonzero(mask)

    A = np.zeros((n, n), dtype=np.float32)
    mr = motifRegular(device=device, numOfNeuron=n)

    for r in tqdm(range(n_rand), desc="Sampling ER", leave=False):
        A.fill(0.0)
        chosen = rng.choice(offdiag_idx, size=e, replace=False)
        A.flat[chosen] = 1.0

        q = mr.cal(torch.from_numpy(A).to(device=device))
        samples[r] = q.cpu().numpy()

    return samples


def analyze_and_plot(A, p=1.0, n_rand=100, threshold=0.0, seed=0, device="cpu"):
    """
    给一个邻接矩阵 A（numpy），计算：
      - ER 随机图下 triad motif 计数的均值 / 标准差
    注意：这里只用到 A 的大小和边数，结构不影响 ER 基线。
    """
    A = np.array(A, dtype=float)
    np.fill_diagonal(A, 0.0)

    n_real = A.shape[0]
    edge_mask = A > threshold
    np.fill_diagonal(edge_mask, False)

    e_real = int(edge_mask.sum())
    p_real = e_real / (n_real * (n_real - 1)) if n_real > 1 else 0.0

    if p < 1.0:
        n_sub = int(round(n_real * p))
        if n_sub < 2:
            raise ValueError("p 太小导致 n_sub < 2")
        e_sub = int(round(p_real * (n_sub * (n_sub - 1))))
    else:
        n_sub = n_real
        e_sub = e_real

    samples = sample_random_counts(n_sub, e_sub, n_rand=n_rand, seed=seed, device=device)
    mu = samples.mean(axis=0)
    sd = samples.std(axis=0, ddof=1)

    idx = [f"Motif {i+1}" for i in range(13)]
    df = pd.DataFrame({
        "ER mean": mu,
        "ER sd":   sd,
    }, index=idx)

    return {
        "table":  df,
        "er_mu":  mu,
        "er_sd":  sd,
        "n":      n_sub,
        "e":      e_sub,
        "samples": samples,
    }


# ================= 主程序：含 MOp 的 clone 合并成一个图，L2 归一化后算 motif =================

if __name__ == "__main__":
    # ===== 1. 路径（看你现在用 2 阈值还是 5 阈值，自己改）=====
    sc_path    = "wb_alltype_sc_subset_zj_1_3_42_1209.npy"
    names_path = "wb_alltype_sc_subset_names_zj_1_3_42_1209.npy"
    csv_path   = "Metadata_PFC_MOp.csv"   # 如果在子文件夹自己补路径

    # ===== 2. 载入数据 =====
    sc = np.load(sc_path)                            # N x N
    names = np.load(names_path, allow_pickle=True)   # 长度 N
    if isinstance(names, np.ndarray):
        names = names.tolist()

    df = pd.read_csv(csv_path)

    # 哪一列对应 names
    if "name" in df.columns:
        neuron_id_col = "name"
    elif "Unnamed: 0" in df.columns:
        neuron_id_col = "Unnamed: 0"
    else:
        raise ValueError("CSV 里找不到和 names 对应的列（尝试了 'name' / 'Unnamed: 0'），手动改 neuron_id_col。")

    if "clone" not in df.columns:
        raise ValueError("CSV 里没有 'clone' 列，不能按 clone 分组。")

    # 哪一列用于判断 MOp（优先用 SomaRegion）
    if "SomaRegion" in df.columns:
        region_col = "SomaRegion"
    elif "Region" in df.columns:
        region_col = "Region"
    else:
        raise ValueError("CSV 里既没有 'SomaRegion' 也没有 'Region'，不知道怎么判断 MOp。")

    # ===== 3. 找出含 MOp 神经元的 clone =====
    mask_mop = df[region_col].astype(str) == "MOp"
    df_mop   = df[mask_mop].copy()

    if df_mop.empty:
        raise ValueError("在 CSV 里没有找到任何 Region/SomaRegion 为 MOp 的神经元。")

    mop_clones = set(df_mop["clone"].astype(str).unique())
    print(f"含 MOp 神经元的 clone 个数: {len(mop_clones)}")
    print("示例几个 clone:", list(mop_clones)[:10])

    # 建立 name -> clone 映射
    df["__name_str__"] = df[neuron_id_col].astype(str)
    name_to_clone = dict(zip(df["__name_str__"], df["clone"].astype(str)))

    # 在 names 里找到：属于这些 clone 的所有 neuron index
    idxs = []
    for i, nid in enumerate(names):
        nid_str = str(nid)
        cl = name_to_clone.get(nid_str, None)
        if cl in mop_clones:
            idxs.append(i)

    print(f"这些含 MOp 的 clone 一共包含神经元数: {len(idxs)}")

    if len(idxs) < 3:
        raise ValueError("整体节点数 < 3，没法算 triad motif。")

    # ===== 4. 取子矩阵，并做行向量 L2 归一化 =====
    A_sub = sc[np.ix_(idxs, idxs)].astype(float)

    # 每行 L2 归一化：每个神经元的出边向量 / 二范数
    row_norms = np.linalg.norm(A_sub, axis=1, keepdims=True)  # shape (n, 1)
    row_norms[row_norms == 0] = 1.0     # 避免除 0
    A_sub_norm = A_sub / row_norms

    # 选一个阈值：归一化后权重 > threshold 视为有边
    # 你可以根据出度分布调整，比如 0.05 / 0.1 / 0.2
    threshold = 0.05

    # 二值邻接矩阵
    W_bin = (A_sub_norm > threshold).astype(np.float32)
    np.fill_diagonal(W_bin, 0.0)

    n = W_bin.shape[0]
    e = int(W_bin.sum())
    max_edges = n * (n - 1)
    sparsity = e / max_edges if max_edges > 0 else np.nan

    print("\n=== 含 MOp 的 clone 合并子图（L2 归一化 + 阈值 {:.3f}） ===".format(threshold))
    print("节点数 n =", n)
    print("边数 e  =", e)
    print("稀疏度 sparsity =", sparsity)

    if e == 0:
        raise ValueError("阈值太高，图里一条边都没有，没法算 motif。请降低 threshold。")

    # ===== 5. 真实网络 13 种 motif 计数 =====
    mr_real = motifRegular(device="cpu", numOfNeuron=n)
    real_counts = mr_real.cal(torch.from_numpy(W_bin)).cpu().numpy()  # shape (13,)

    # ===== 6. ER baseline（在同样的 n、e 下随机图的 motif 统计）=====
    # 注意：这里传的是 W_bin（只用到 n 和 e_real）
    er_res = analyze_and_plot(
        W_bin,
        p=1.0,
        n_rand=200,       # 采样次数，想稳一点可以调大
        threshold=0.0,    # W_bin 里 0/1，用 >0 判边即可
        seed=42,
        device="cpu",
    )
    mu = er_res["er_mu"]
    sd = er_res["er_sd"]

    # ===== 7. NZ-score =====
    z = np.zeros_like(real_counts)
    valid = sd > 1e-8
    z[valid] = (real_counts[valid] - mu[valid]) / sd[valid]

    motif_labels = [f"Motif {i+1}" for i in range(13)]
    df_motif = pd.DataFrame({
        "real": real_counts,
        "ER_mean": mu,
        "ER_sd": sd,
        "NZ_ER": z,
    }, index=motif_labels)

    print("\n=== 含 MOp 的 clone（所有相关神经元合在一起）的 motif 分布（L2 归一化后） ===")
    print(df_motif.round(2))

    # 存一份结果
    df_motif.to_csv("motif_MOp_related_clones_L2norm.csv")